# ESM-2 Viral - Domain adaption on viral dataset

In [ ]:
wget https://rvdb-prot.pasteur.fr/files/U-RVDBv30.0-prot_clustered.fasta.xz -O data/raw/CRVDBv30-prot.fasta.xz
xz -d data/raw/CRVDBv30-prot.fasta.xz

Total sequences: 785890
Valid sequences: 658064
23991 sequences were longer than 1600.
Invalid characters found in 103835 sequences.
Split completed: 592257 train, 32903 val, 32904 test sequences.

In [ ]:
wget https://rvdb-prot.pasteur.fr/files/U-RVDBv30.0-prot.fasta.xz -O data/raw/URVDBv30prot.fasta.xz
xz -d data/raw/URVDBv30prot.fasta.xz

In [ ]:
# remove duplicates from fasta file
seqkit rmdup -s -j 28 data/raw/URVDBv30prot.fasta > data/raw/URVDBv30prot_rmdup.fasta

Total sequneces in the U-RVDB: 41326058         
Remaing sequences after filter: 5535631    

In [ ]:
# preprocess will filter non canonical AA, plus seques longer than 1600.
python src/data/preprocess.py

Out of a total of 5535631 sequences:       
2007048 sequences were longer than 1600.      
2696018 were Valid sequences.   

# Adding Eucariote and prokariotes sequences collected from uniprot

In [ ]:
# munber of sequences: 551,657
NOT (taxonomy_id:10239) AND (length:[* TO 1600]) AND (reviewed:true).     

## cluster the data to 50% identify

Program: CD-HIT, V4.8.1
Command: cd-hit -i
         uniprotkb_NOT_taxonomy_id_10239_AND_len_1600.fasta.gz
         -o
         uniprotkb_NOT_taxonomy_id_10239_AND_len_1600_Clustered0.5.fasta
         -c 0.5 -n 2 -T 100 -M 0

In [2]:
2599411 - 2426416

172995

## Preprocessing the fasta file for domain adaptation

In [ ]:
import os
from Bio import SeqIo

In [ ]:
#input_fasta = "../data/raw/URVDBv29-prot_clustered.fasta"
input_fasta = "../data/processed/C-RVDBv29_no_poly/train.fasta"

records = list(SeqIO.parse(input_fasta, "fasta"))

#records = [record for record in records if 'poly' in record.description]
len(records)

568485

In [7]:
mn=0
mx=0
for record in records:
    if len(record.seq) < mn or mn == 0:
        mn = len(record.seq)
    if len(record.seq) > mx:
        mx = len(record.seq)
print(f"Min length: {mn}")
print(f"Max length: {mx}")


Min length: 11
Max length: 2047


In [ ]:
count = 0
while count < 10:
    record = records[count]
    print(record.description)
    count += 1

acc|GENBANK|UJJ65121.1|GENBANK|OM336640|surface glycoprotein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65122.1|GENBANK|OM336640|ORF3a protein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65123.1|GENBANK|OM336640|envelope protein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65124.1|GENBANK|OM336640|membrane glycoprotein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65125.1|GENBANK|OM336640|ORF6 protein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65126.1|GENBANK|OM336640|ORF7a protein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65127.1|GENBANK|OM336640|ORF7b [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65128.1|GENBANK|OM336640|ORF8 protein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65129.1|GENBANK|OM336640|nucleocapsid phosphoprotein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65130.1|GENBANK|OM336640|OR

In [ ]:
data_dir = '../data/viral/mutant_sequences'

for file in os.listdir(data_dir):
    records = list(SeqIO.parse(os.path.join(data_dir, file), "fasta"))
    if len(records[0].seq) > 1022:
        print(records[0].id, len(records[0].seq))

SARS2_RBD_N331C 1273
CVB3_POLG_M1D 2185
EV_REP_G1I 1331
SARS2_DELTA_M1F 1250
SARS2_RBD_N331C 1273
SARS2_BA1_M1I 1249


In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

def filter_fasta_by_valid_amino_acids(input_fasta_path, output_fasta_path):
    """
    Filters a FASTA file to keep only sequences with valid amino acids.

    Args:
        input_fasta_path (str): Path to the input FASTA file.
        output_fasta_path (str): Path to save the filtered FASTA file.
    """
    valid_amino_acids = set("ACDEFGHIKLMNPQRSTVWY")
    
    valid_records = []
    for record in SeqIO.parse(input_fasta_path, "fasta"):
        sequence_str = str(record.seq).upper()
        is_valid = True
        for aa in sequence_str:
            if aa not in valid_amino_acids:
                is_valid = False
                print(f"Invalid character '{aa}' found in sequence: {record.id}. Skipping.")
                break
        if is_valid:
            valid_records.append(record)

    SeqIO.write(valid_records, output_fasta_path, "fasta")
    print(f"Filtered FASTA saved to: {output_fasta_path}")
    print(f"Original records: {len(list(SeqIO.parse(input_fasta_path, 'fasta')))}, Valid records: {len(valid_records)}")

# Example usage:
input_file = "../data/processed/C-RVDBv29_no_poly/train.fasta"
output_file = "../data/processed/C-RVDBv29_no_poly_20aa/train.fasta"
filter_fasta_by_valid_amino_acids(input_file, output_file)

In [4]:
[x for x in ['vicam_300m', 'vicam_600m','esmc_300m', 'esmc_600m'] if x in '/chpc/home/C-RVDBv29_no_poly/vicam_600m.fasta'][0]

'vicam_600m'

# Dataset ID

In [12]:
import os

In [10]:
print([ord(char) for char in 'Hello world!'])
print(sum([ord(char) for char in 'Hello_world 2015!']))

[72, 101, 108, 108, 111, 32, 119, 111, 114, 108, 100, 33]
1412


In [1]:
def string_to_number(s):
    return sum(ord(char) for char in s)

In [23]:
output = "experiments/lassoCV_zscore-1_filtered/esm2_650m/cellular/pool_split/BLAT_ecoli.csv"
file = os.path.basename(output).replace('_', '').replace('.csv', '')
file

'BLATecoli'

In [24]:
string_to_number(file)

815

In [ ]:
import hashlib

def make_seed(dataset_id, rep):
    s = f"{dataset_id}_{rep}"
    h = hashlib.md5(s.encode()).hexdigest()[:8]
    return int(h, 16)


output = 'AMIE_PSEAE_Whitehead'
ds = output.split('/')[-1].split('.csv')[0] 
for i in [1, 2, 3]:
    print(make_seed(ds, i))



1961240317
3165731295
86658944


# Site split

In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('../data/cellular/metadata/AMIE_PSEAE_Whitehead.csv')
df

,Unnamed: 0,ID,mutant,target,sequence
0,0,AMIE_PSEAE_M1W,M1W,-0.5174,WRHGDISSSNDTVGVAVVNYKMPRLHTAAEVLDNARKIAEMIVGMK...
1,1,AMIE_PSEAE_M1Y,M1Y,-0.5253,YRHGDISSSNDTVGVAVVNYKMPRLHTAAEVLDNARKIAEMIVGMK...
2,2,AMIE_PSEAE_M1P,M1P,-0.5154,PRHGDISSSNDTVGVAVVNYKMPRLHTAAEVLDNARKIAEMIVGMK...
3,3,AMIE_PSEAE_M1M,M1M,0.0000,MRHGDISSSNDTVGVAVVNYKMPRLHTAAEVLDNARKIAEMIVGMK...
4,4,AMIE_PSEAE_M1I,M1I,-0.3640,IRHGDISSSNDTVGVAVVNYKMPRLHTAAEVLDNARKIAEMIVGMK...
...,...,...,...,...,...
6563,6814,AMIE_PSEAE_G341D,G341D,0.0331,MRHGDISSSNDTVGVAVVNYKMPRLHTAAEVLDNARKIAEMIVGMK...
6564,6815,AMIE_PSEAE_G341E,G341E,-0.0263,MRHGDISSSNDTVGVAVVNYKMPRLHTAAEVLDNARKIAEMIVGMK...
6565,6816,AMIE_PSEAE_G341H,G341H,-0.0578,MRHGDISSSNDTVGVAVVNYKMPRLHTAAEVLDNARKIAEMIVGMK...
6566,6817,AMIE_PSEAE_G341K,G341K,-0.1361,MRHGDISSSNDTVGVAVVNYKMPRLHTAAEVLDNARKIAEMIVGMK...


In [ ]:
def split_data(df, seed, test_size=0.2):
    """
    This function randomly splits a dataframe into train, validation data given a seed by mutation site.
    Parameters:
     - df (DataFrame): dataframe containing information about mutants. Mutants should be in the order wt amino acid, site of mutation, mutant amino acid. ex "M1F"
     - seed (int): the seed to be used when shuffling sites randomly.
     - test_size (float): the percentage of data that will be split into the validation dataset. Default is 0.2

    Returns:
     - train_df (DataFrame): the DataFrame containing selected data by site to be used as the train dataset.
     - val_df (DataFrame): the DataFrame containing selected data by site to be used as the validation dataset.
    """
    
    # Extract mutation sites
    df = df.copy()
    df["site"] = df["mutant"].str.extract(r'(\d+)').astype(int)
    
    # Get unique sites and shuffle
    sites = df["site"].unique().tolist()
    rng = np.random.default_rng(seed)
    rng.shuffle(sites)
    
    # Calculate split point based on number of rows
    cumsum_rows = 0
    target_val_rows = len(df) * test_size
    val_sites = []
    
    for site in sites:
        site_row_count = (df["site"] == site).sum()
        if cumsum_rows < target_val_rows:
            val_sites.append(site)
            cumsum_rows += site_row_count
        else:
            break
    
    # Split dataframe
    val_df = df[df["site"].isin(val_sites)]
    train_df = df[~df["site"].isin(val_sites)]

    if not any(site in train_df['site'].unique() for site in val_df['site'].unique()):
        print('All sites are unique for train or test.')
        
    return train_df, val_df

In [18]:
train, test = split_data(df, 42)

All sites are unique for train or test.


In [7]:
train

,Unnamed: 0,ID,mutant,target,sequence,site
19,19,AMIE_PSEAE_R2F,R2F,-0.3240,MFHGDISSSNDTVGVAVVNYKMPRLHTAAEVLDNARKIAEMIVGMK...,2
20,21,AMIE_PSEAE_R2Y,R2Y,-0.5054,MYHGDISSSNDTVGVAVVNYKMPRLHTAAEVLDNARKIAEMIVGMK...,2
21,22,AMIE_PSEAE_R2P,R2P,-0.7597,MPHGDISSSNDTVGVAVVNYKMPRLHTAAEVLDNARKIAEMIVGMK...,2
22,23,AMIE_PSEAE_R2M,R2M,-0.2318,MMHGDISSSNDTVGVAVVNYKMPRLHTAAEVLDNARKIAEMIVGMK...,2
23,24,AMIE_PSEAE_R2I,R2I,-0.4327,MIHGDISSSNDTVGVAVVNYKMPRLHTAAEVLDNARKIAEMIVGMK...,2
...,...,...,...,...,...,...
6563,6814,AMIE_PSEAE_G341D,G341D,0.0331,MRHGDISSSNDTVGVAVVNYKMPRLHTAAEVLDNARKIAEMIVGMK...,341
6564,6815,AMIE_PSEAE_G341E,G341E,-0.0263,MRHGDISSSNDTVGVAVVNYKMPRLHTAAEVLDNARKIAEMIVGMK...,341
6565,6816,AMIE_PSEAE_G341H,G341H,-0.0578,MRHGDISSSNDTVGVAVVNYKMPRLHTAAEVLDNARKIAEMIVGMK...,341
6566,6817,AMIE_PSEAE_G341K,G341K,-0.1361,MRHGDISSSNDTVGVAVVNYKMPRLHTAAEVLDNARKIAEMIVGMK...,341


In [14]:
any(site in train['site'].unique() for site in test['site'].unique())

False

# Creating mutant fasta file

In [1]:
import os
import pandas as pd

In [27]:
base_dir = '../data/viral/metadata/'
for file in os.listdir(base_dir):
    df = pd.read_csv(os.path.join(base_dir, file))
    df['ID'] = df['ID'].apply(lambda x: '_'.join(x.split('_')[:-1])) + '_' + df['mutant']
    df.to_csv(os.path.join(base_dir, file), index=False)

df

,ID,mutant,num_mutations,target,sequence
0,SARS2_RBD_N331C,N331C,1,-1.26,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
1,SARS2_RBD_N331D,N331D,1,-0.44,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
2,SARS2_RBD_N331A,N331A,1,-0.11,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
3,SARS2_RBD_N331Y,N331Y,1,-1.02,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
4,SARS2_RBD_N331W,N331W,1,-1.12,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
...,...,...,...,...,...
3793,SARS2_RBD_T531F,T531F,1,-0.02,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
3794,SARS2_RBD_T531E,T531E,1,0.03,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
3795,SARS2_RBD_T531D,T531D,1,0.03,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
3796,SARS2_RBD_T531A,T531A,1,-0.02,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...


In [28]:
base_dir = '../data/viral/metadata/'
for file in os.listdir(base_dir):
    df = pd.read_csv(os.path.join(base_dir, file))
    file_name = file.strip('.csv')

    with open(f"../data/viral/mutant_sequences/{file_name}.fasta", "w") as f:
        for _, row in df.iterrows():
            f.write(f">{row['ID']}\n{row['sequence']}\n")


## creating a fasta file loader

In [31]:
class FastaDataLoader:
    """
    Data loader for reading a FASTA file and creating batches based on a token limit.
    
    Args:
    - fasta_file (str): Path to the FASTA file.
    - batch_token_limit (int, optional): Maximum number of tokens per batch. Defaults to 4096.
    - model (object): Model object with a `_tokenize` method for tokenizing sequences.
    """
    def __init__(self, fasta_file, batch_token_limit=4096):
        self.fasta_file = fasta_file
        self.batch_token_limit = batch_token_limit
        self.sequences = list(SeqIO.parse(fasta_file, "fasta"))
        self.total_sequences = len(self.sequences)
        
        # Check for duplicate sequence labels
        sequence_labels = [seq.id for seq in self.sequences]
        assert len(set(sequence_labels)) == len(sequence_labels), "Found duplicate sequence labels"

    def __len__(self):
        # Approximate total number of batches
        total_tokens = sum(len(str(seq.seq)) + 2 for seq in self.sequences)  # +2 for BOS and EOS tokens
        return (total_tokens + self.batch_token_limit - 1) // self.batch_token_limit

    def __iter__(self):
        ids, lengths, seqs = [], [], []
        current_token_count = 0

        for seq in self.sequences:
            seq_length = len(seq.seq)
            token_count = seq_length + 2  # Include BOS and EOS tokens
            if current_token_count + token_count > self.batch_token_limit and ids:
                # Yield current batch if adding the new sequence exceeds the token limit
                yield ids, lengths, seqs
                ids, lengths, seqs = [], [], []
                current_token_count = 0

            # Add the current sequence to the batch
            ids.append(seq.id)
            lengths.append(seq_length)
            seqs.append(str(seq.seq))
            current_token_count += token_count

        # Yield any remaining sequences
        if ids:
            yield ids, lengths, seqs

In [34]:
from Bio import SeqIO
from tqdm import tqdm
fasta_file = "../test.fasta"
data_loader = FastaDataLoader(fasta_file)
    
for batch_ids, batch_lengths, batch_seqs in tqdm(data_loader, desc="Processing batches", leave=False):
    print(f"Batch IDs: {batch_ids}")
    print(f"Batch Lengths: {batch_lengths}")
    print(f"Batch Sequences: {batch_seqs}")
    # Here you can process the batch with your model

Batch IDs: ['a4', 'a6', 'a10']
Batch Lengths: [4, 6, 10]
Batch Sequences: ['AAAA', 'AAAAAA', 'AAAAAAAAAA']


# rename columns in metadata

In [6]:
import pandas as pd
import os


In [36]:
folder = '../data/viral/metadata/'

cols = ['ID','mutant','num_mutations','target','sequence']
for file in os.listdir(folder):
    if file.endswith('_metadata.csv'):
        df = pd.read_csv(os.path.join(folder, file))
        dts_name = file.split('_metadata.csv')[0]
        #length = [len(x.split(':')) for x in df['ID']]
        #print(f"{dts_name}: max length: {max(length)}")
        df['num_mutations'] = [len(x.split(':')) for x in df['mutant']]
        ID = dts_name.split('_')
        df['ID'] = [f"{ID[0]}_{ID[1]}_{i+1}" for i in range(len(df))]
        file_path = os.path.join(folder, dts_name + '.csv')
        df[cols].to_csv(file_path, index=False)
        print(f"Saving {file_path}")
        

df[cols]

Saving ../data/viral/metadata/LASSA_GP_Carr.csv
Saving ../data/viral/metadata/CVB3_3D_Alvarez.csv
Saving ../data/viral/metadata/IAV_H5_HA_Dadonaite.csv
Saving ../data/viral/metadata/PESV_POLG_Tsuboyama.csv
Saving ../data/viral/metadata/SARS2_BA1_SPIKE_Dadonaite.csv
Saving ../data/viral/metadata/HIV1_HV1B9_ENV_DuenasDecamp.csv
Saving ../data/viral/metadata/SARS2_DELTA_SPIKE_Dadonaite.csv
Saving ../data/viral/metadata/CVB3_3B_Alvarez.csv
Saving ../data/viral/metadata/SARS2_XBB15_RBD_Taylor.csv
Saving ../data/viral/metadata/SARS2_RBD_Starr_binding.csv
Saving ../data/viral/metadata/CVB3_2C_Alvarez.csv
Saving ../data/viral/metadata/SARS2_PRD0038_RBD_Starr.csv
Saving ../data/viral/metadata/DENV_POLG_Suphatrakul.csv
Saving ../data/viral/metadata/SARS2_RBD_Starr_expression.csv
Saving ../data/viral/metadata/IAV_H3_NP_Doud.csv
Saving ../data/viral/metadata/BPP22_COAT_Tsuboyama.csv
Saving ../data/viral/metadata/IAV_H1_HA_Doud.csv
Saving ../data/viral/metadata/CVB3_2B_Alvarez.csv
Saving ../data/vi

,ID,mutant,num_mutations,target,sequence
0,BP434_RPC1_1,A18C,1,-0.167281,SISSRVKSKRIQLGLNQCELAQKVGTTQQSIEQLENGKTKRPRFLP...
1,BP434_RPC1_2,A18D,1,-0.238248,SISSRVKSKRIQLGLNQDELAQKVGTTQQSIEQLENGKTKRPRFLP...
2,BP434_RPC1_3,A18E,1,-0.034313,SISSRVKSKRIQLGLNQEELAQKVGTTQQSIEQLENGKTKRPRFLP...
3,BP434_RPC1_4,A18F,1,0.121522,SISSRVKSKRIQLGLNQFELAQKVGTTQQSIEQLENGKTKRPRFLP...
4,BP434_RPC1_5,A18G,1,-0.490995,SISSRVKSKRIQLGLNQGELAQKVGTTQQSIEQLENGKTKRPRFLP...
...,...,...,...,...,...
1454,BP434_RPC1_1455,W58R,1,-3.041730,SISSRVKSKRIQLGLNQAELAQKVGTTQQSIEQLENGKTKRPRFLP...
1455,BP434_RPC1_1456,W58S,1,-2.247879,SISSRVKSKRIQLGLNQAELAQKVGTTQQSIEQLENGKTKRPRFLP...
1456,BP434_RPC1_1457,W58T,1,-2.544818,SISSRVKSKRIQLGLNQAELAQKVGTTQQSIEQLENGKTKRPRFLP...
1457,BP434_RPC1_1458,W58V,1,-2.341668,SISSRVKSKRIQLGLNQAELAQKVGTTQQSIEQLENGKTKRPRFLP...


In [14]:
df = pd.read_csv('../data/marks_data/viral_metadata/AAV2_CAPSD_Sinai_metadata.csv', index_col=0)
df['num_muts'] = [len(x.split(':')) for x in df['ID']]
df.query('num_muts == 28')

,ID,mutant,target,sequence,num_muts
1794,D561C:E562S:E563L:E564C:I565L:R566H:T567M:T568...,D561C:E562S:E563L:E564C:I565L:R566H:T567M:T568...,-3.708075,MAADGYLPDWLEDTLSEGIRQWWKLKPGPPPPKPAERHKDDSRGLV...,28
2988,D561H:E562I:E563D:E564N:I565C:R566C:T567A:T568...,D561H:E562I:E563D:E564N:I565C:R566C:T567A:T568...,-3.832230,MAADGYLPDWLEDTLSEGIRQWWKLKPGPPPPKPAERHKDDSRGLV...,28
6856,D561S:E562V:E563F:E564V:I565V:R566N:T567A:T568...,D561S:E562V:E563F:E564V:I565V:R566N:T567A:T568...,-3.224306,MAADGYLPDWLEDTLSEGIRQWWKLKPGPPPPKPAERHKDDSRGLV...,28


In [16]:
df.iloc[1794]['ID']

'D561C:E562S:E563L:E564C:I565L:R566H:T567M:T568I:N569L:P570F:V571Y:A572M:T573H:E574C:Q575C:Y576Q:G577F:S578I:V579L:S580Q:T581G:N582T:L583V:Q584L:R585I:G586S:N587A:R588H'

In [1]:
def split_data(df, seed, train_pct=0.8, val_pct=0.2):
    """
    This function randomly splits a dataframe into train, validationi data given a seed by mutation site.
    Parameters:
     - df (DataFrame): dataframe containing information about mutants. Mutants should be in the order wt amino acid, site of mutation, mutant amino acid. ex "M1F"
     - seed (int): the seed to be used when shuffling sites randomly.
     - train_pct (float): the percentage of data that will be split into the train dataset. Default is 0.8
     - val_pct (float): the percentage of data that will be split into the validation dataset. Default is 0.2

    Returns:
     - train_df (DataFrame): the DataFrame containing selected data by site to be used as the train dataset.
     - val_df (DataFrame): the DataFrame containing selected data by site to be used as the validation dataset.
    """
    # find sites of mutation and order randomly
    df["site"] = [int(s[1:-1]) for s in df["mutant"]]
    sites = df["site"].unique()
    random.seed(seed)
    random.shuffle(sites)

    if train_pct + val_pct != 1:
        print("Split percentages must sum to 1")
        return

    df_size = df.shape[0]
    df_val_size = df_size*val_pct
    val_sites, train_sites = [], []

    # determine sites for validation, then train
    for site in sites:
        if len(val_sites) <= df_val_size:
            val_sites.extend([mut_site for mut_site in df["site"] if mut_site == site])
        else:
            train_sites.extend([mut_site for mut_site in df["site"] if mut_site == site])

    # subset df for train, test data
    train_df = df[df["site"].isin(set(train_sites))]
    val_df = df[df["site"].isin(set(val_sites))]

    return train_df, val_df

# data normality

In [ ]:
import os
import scipy
import torch
import argparse
import numpy as np
import pandas as pd
from scipy import stats
from sklearn import metrics
from sklearn.linear_model import Lasso, LassoCV
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from scipy.stats import spearmanr
import random
import plotnine as pln

In [ ]:
scaler = StandardScaler()

data_dir = '../data/nonviral/metadata/'
nonviral = pd.DataFrame()

for file in os.listdir(data_dir):
    df = pd.read_csv(os.path.join(data_dir, file))
    df['Dataset'] = file.strip('.csv')
    df['target'] = scaler.fit_transform(df['target'].to_frame()).squeeze()

    nonviral = pd.concat([nonviral, df])

nonviral.head()

In [ ]:
plot = (
    pln.ggplot(nonviral, pln.aes(y='target', x='Dataset')) +
    pln.geom_boxplot() +
    pln.theme(figure_size=(12,6), axis_text_x=pln.element_text(rotation=90))
    )
plot

In [ ]:
plot = (
    pln.ggplot(nonviral, pln.aes(x='target')) +
    pln.geom_density() +
    pln.facet_wrap('~Dataset', scales='free_y', ncol=3) +
    pln.theme(figure_size=(8,12))
    )
plot

In [ ]:
data_dir = '../data/viral/metadata/'
viral = pd.DataFrame()

for file in os.listdir(data_dir):
    df = pd.read_csv(os.path.join(data_dir, file))
    df['Dataset'] = file.strip('.csv')

    viral = pd.concat([viral, df])


plot = (
    pln.ggplot(viral, pln.aes(x='target')) +
    pln.geom_density() +
    pln.facet_wrap('~Dataset', scales='free_y', ncol=3) +
    pln.theme(figure_size=(8,12))
    )
plot

In [ ]:
plot = (
    pln.ggplot(viral.query('Dataset != "SARS2_DELTA_SPIKE_Dadonaite"'), pln.aes(y='target', x='Dataset')) +
    pln.geom_boxplot() +
    pln.theme(figure_size=(12,6), axis_text_x=pln.element_text(rotation=90))
    )
plot

# Split stretegies

## Split datasets by site

In [ ]:
import pandas as pd
import random

In [ ]:
data = pd.read_csv('../data/nonviral/metadata/PABP_YEAST_Fields2013_doubles_metadata.csv', index_col=0)
data

In [ ]:
data['mutant'].str.extract(r'(\d+)').dropna()

In [ ]:
df = pd.read_csv('../data/nonviral/metadata/PTEN_HUMAN_Fowler2018_metadata.csv', index_col=0)
df

In [ ]:
sites = df['mutant'].str.extract(r'(\d+)').dropna()
sites[0].unique().astype(int)

In [ ]:
df = pd.read_csv('../data/nonviral/metadata/TIM_SULSO_metadata.csv', index_col=0)
print(df.shape)
train_pct=0.8
test_pct=0.2
seed = 42

df["site"] = [int(s[1:-1]) for s in df["mutant"]]
sites = df["site"].unique()
random.seed(seed)
random.shuffle(sites)

if train_pct + test_pct != 1:
    print("Split percentages must sum to 1")
  
df_size = df.shape[0]
df_test_size = df_size*test_pct
test_sites, train_sites = [], []

print(df_size, df_test_size)
# determine sites for test, then train
for site in sites:
    if len(test_sites) <= df_test_size:
        test_sites.extend([mut_site for mut_site in df["site"] if mut_site == site])
    else:
        train_sites.extend([mut_site for mut_site in df["site"] if mut_site == site])


# subset df for train, test data
train_df = df[df["site"].isin(set(train_sites))]
test_df = df[df["site"].isin(set(test_sites))]
test_df

In [ ]:
def split_data(df, seed, train_pct=0.85, test_pct=0.15):
    """
    This function randomly splits a dataframe into train, test, and validation data given a seed by mutation site.
    
    Parameters:
     - df (DataFrame): dataframe containing information about mutants. Mutants should be in the order wt amino acid, site of mutation, mutant amino acid. ex "M1F"
     - seed (int): the seed to be used when shuffling sites randomly.
     - train_pct (float): the percentage of data that will be split into the train dataset. Default is 0.8
     - test_pct (float): the percentage of data that will be split into the test dataset. Default is 0.2

    Returns:
     - train_df (DataFrame): the DataFrame containing randomly selected data by site to be used as the train dataset.
     - test_df (DataFrame): the DataFrame containing randomly selected data by site to be used as the test dataset.
     - val_df (DataFrame): the DataFrame containing randomly selected data by site to be used as the val dataset.
    """
    # find sites of mutation and order randomly
    df["site"] = [int(s[1:-1]) for s in df["mutant"]]
    sites = df["site"].unique()
    random.seed(seed)
    random.shuffle(sites)

    if train_pct + test_pct != 1:
        print("Split percentages must sum to 1")
        return

    df_size = df.shape[0]
    df_test_size = df_size*test_pct
    test_sites, train_sites = [], []

    # determine sites for test, then train
    for site in sites:
        if len(test_sites) <= df_test_size:
            test_sites.extend([mut_site for mut_site in df["site"] if mut_site == site])
        else:
            train_sites.extend([mut_site for mut_site in df["site"] if mut_site == site])

    # subset df for train, test data
    train_df = df[df["site"].isin(set(train_sites))]
    test_df = df[df["site"].isin(set(test_sites))]

    return train_df, test_df


In [ ]:
path_meta_data = '../data/nonviral/metadata/TPMT_HUMAN_Fowler2018.csv'
meta_data = pd.read_csv(path_meta_data)
meta_data = meta_data.query("mutant != 'WT'")
meta_data

In [ ]:
scaler = StandardScaler()
meta_data['target'] = scaler.fit_transform(meta_data['target'].to_frame()).squeeze()
meta_data

In [ ]:
#meta_data[meta_data['mutant'].str.contains(r'^[A-Z]\+[0-9]{1,3}[A-Z]$')]
meta_data[meta_data['mutant'].str.contains('wt')]


In [ ]:
# load and merge the data with features
path_compressed_embed_file = '../embeddings/esm2_650m/nonviral/TPMT_HUMAN_Fowler2018.pt'
embed = torch.load(path_compressed_embed_file, weights_only=True)
embed_df = pd.DataFrame.from_dict(embed).T.reset_index()
#embed_df = features_scaler(embed_df)#.reset_index()
embed_df.rename(columns={'index': 'ID'}, inplace=True)
embed_df

In [ ]:
train, test = split_data(meta_data, 420)
        
train_data = train.merge(embed_df, how='inner', left_on='ID', right_on='ID')
test_data = test.merge(embed_df, how='inner', left_on='ID', right_on='ID')

y_train = train_data['target']
y_test = test_data['target']

X_train = train_data.iloc[:, train.shape[1]:]
X_test = test_data.iloc[:, test.shape[1]:]
print(f'X train shape: {X_train.shape}, X test shape: {X_test.shape}')

In [ ]:
model = LassoCV(max_iter=10000, tol=1e-4, n_jobs=-1)
model.fit(X_train, y_train)

In [ ]:
# Make predictions
y_pred_train = pd.DataFrame(model.predict(X_train))
y_pred_test = pd.DataFrame(model.predict(X_test))

# Evaluate the model
r2_train = metrics.r2_score(y_train, y_pred_train)
mae_train = metrics.mean_absolute_error(y_train, y_pred_train)
mse_train = metrics.mean_squared_error(y_train, y_pred_train)
rmse_train = np.sqrt(mse_train)
rho_train, p_value_train = spearmanr(y_train, y_pred_train)

r2_test = metrics.r2_score(y_test, y_pred_test)
mae_test = metrics.mean_absolute_error(y_test, y_pred_test)
mse_test = metrics.mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)
rho_test, p_value_test = spearmanr(y_test, y_pred_test)

print(f"Results: r2_train: {r2_train:.3f}, r2_test: {r2_test:.3f}")

# protein features

In [ ]:
# Create dictionary to store both metrics
prot_data = {}

# Viral proteins
for prot in os.listdir('../data/viral/metadata/'):
    if prot.endswith('.csv'):
        df = pd.read_csv(os.path.join('../data/viral/metadata/', prot), index_col=0)
        prot_name = prot.split('.csv')[0]
        
        prot_data[prot_name] = {
            'avg_length': int(df['sequence'].apply(lambda x: len(x)).mean()),
            'sample_size': df.shape[0]
        }

# Create DataFrame
viral_proteins_df = pd.DataFrame.from_dict(prot_data, orient='index').reset_index(drop=False)
viral_proteins_df = viral_proteins_df.sort_values('sample_size')

In [ ]:
viral_proteins_df.query('avg_length > 1300')

In [ ]:
viral_proteins_df.query('avg_length < 1022 & sample_size > 2000 & index not in @doubles')#['index'].values

In [ ]:
# Create dictionary to store both metrics
prot_data = {}

# cellular proteins
for prot in os.listdir('../data/cellular/metadata/'):
    if prot.endswith('.csv'):
        df = pd.read_csv(os.path.join('../data/cellular/metadata/', prot), index_col=0)
        prot_name = prot.split('.csv')[0]
        
        prot_data[prot_name] = {
            'avg_length': int(df['sequence'].apply(lambda x: len(x)).mean()),
            'sample_size': df.shape[0]
        }

# Create DataFrame
cellular_proteins_df = pd.DataFrame.from_dict(prot_data, orient='index').reset_index(drop=False)
cellular_proteins_df = cellular_proteins_df.sort_values('sample_size')

In [ ]:
cellular_proteins_df.query('avg_length < 1022 & sample_size > 2000 & index not in @doubles')#['index'].values
#cellular_proteins_df.sort_values('avg_length')